In [55]:
import re

import pandas as pd
import numpy as np

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, f1_score, classification_report

In [56]:
# ---- LOAD ----
# Structured sentence data
#df = pd.read_csv("../data/generated_sentences.csv").dropna(subset=["sentence", "label"])

# Unstructured sentence data
df = pd.read_csv("../data/filtered_sentences.csv").dropna(subset=["sentence", "label"])
df["label"] = df["label"].apply(lambda x: x.strip().split()[-1]) # TAKE LAST WORD AS LABEL
X_raw = df["sentence"].astype(str).tolist()
y_raw = df["label"].astype(str).tolist()

# ---- CHARACTER TOKENIZER ----
all_chars = sorted(set("".join(X_raw)))
char2idx = {c: i+2 for i, c in enumerate(all_chars)}
char2idx["<PAD>"] = 0
char2idx["<UNK>"] = 1

MAX_LEN = 50

In [ ]:
def encode(sentence):
    ids = [char2idx.get(c, 1) for c in sentence]
    ids = ids[:MAX_LEN]
    ids += [0] * (MAX_LEN - len(ids))
    return ids

# ---- LABEL ENCODER ----
le = LabelEncoder()
y_encoded = le.fit_transform(y_raw)
num_classes = len(le.classes_)
label_order = list(le.classes_)

X = [encode(s) for s in X_raw]
X_train, X_test, y_train, y_test = train_test_split(X, y_encoded, test_size=0.2, random_state=42)

# ---- SAMPLE WEIGHTS ----
def get_sample_weight(sentence, label, rule_boost=2.0):
    rules = {
        "PAST":     [r'[аэоөүию]н$', r'сан$|сэн$|сон$|сөн$'],
        "HABITUAL": [r'даг$|дэг$|дог$|дөг$'],
        "FUTURE":   [r'на$|нэ$|но$|нө$'],
    }
    for pattern in rules.get(label, []):
        if re.search(pattern, sentence.strip(), re.IGNORECASE):
            return rule_boost
    return 1.0

train_indices = list(range(len(X_train)))
train_weights = [
    get_sample_weight(X_raw[i], y_raw[i])
    for i in range(len(X_train))
]

# ---- DATASET ----
class SentenceDataset(Dataset):
    def __init__(self, X, y, weights=None):
        self.X = torch.tensor(X, dtype=torch.long)
        self.y = torch.tensor(y, dtype=torch.long)
        self.weights = torch.tensor(weights, dtype=torch.float) if weights else torch.ones(len(X))

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx], self.weights[idx]

train_loader = DataLoader(SentenceDataset(X_train, y_train, train_weights), batch_size=16, shuffle=True)
test_loader  = DataLoader(SentenceDataset(X_test,  y_test),  batch_size=16)

# ---- TRANSFORMER MODEL ----
class TenseTransformer(nn.Module):
    def __init__(self, vocab_size, embed_dim, num_heads, num_layers, num_classes, dropout=0.3):
        super().__init__()
        self.embed     = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.pos_embed = nn.Embedding(MAX_LEN, embed_dim)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim,
            nhead=num_heads,
            dim_feedforward=embed_dim * 4,
            dropout=dropout,
            batch_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.classifier  = nn.Linear(embed_dim, num_classes)
        self.dropout     = nn.Dropout(dropout)

    def forward(self, x):
        positions    = torch.arange(x.size(1), device=x.device).unsqueeze(0)
        x            = self.embed(x) + self.pos_embed(positions)
        x            = self.dropout(x)
        padding_mask = (x.sum(dim=-1) == 0)
        x            = self.transformer(x, src_key_padding_mask=padding_mask)
        x            = x.mean(dim=1)
        return self.classifier(x)

model = TenseTransformer(
    vocab_size  = len(char2idx),
    embed_dim   = 64,
    num_heads   = 4,
    num_layers  = 2,
    num_classes = num_classes
)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss(reduction="none")

# ---- TRAIN ----
EPOCHS = 30

for epoch in range(EPOCHS):
    model.train()
    total_loss, correct = 0, 0

    for X_batch, y_batch, w_batch in train_loader:
        optimizer.zero_grad()
        out  = model(X_batch)
        loss = criterion(out, y_batch)
        loss = (loss * w_batch).mean()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        correct    += (out.argmax(1) == y_batch).sum().item()

    acc = correct / len(X_train)
    print(f"Epoch {epoch+1}/{EPOCHS} — Loss: {total_loss:.4f} — Train Acc: {acc:.2%}")

Epoch 1/30 — Loss: 26.0432 — Train Acc: 50.29%
Epoch 2/30 — Loss: 24.2177 — Train Acc: 52.87%
Epoch 3/30 — Loss: 21.5713 — Train Acc: 58.05%
Epoch 4/30 — Loss: 19.3004 — Train Acc: 60.63%
Epoch 5/30 — Loss: 17.4279 — Train Acc: 69.25%
Epoch 6/30 — Loss: 17.7982 — Train Acc: 65.23%
Epoch 7/30 — Loss: 16.6694 — Train Acc: 64.94%
Epoch 8/30 — Loss: 16.0059 — Train Acc: 68.10%
Epoch 9/30 — Loss: 15.4997 — Train Acc: 69.25%
Epoch 10/30 — Loss: 16.0852 — Train Acc: 70.11%
Epoch 11/30 — Loss: 15.1725 — Train Acc: 69.54%
Epoch 12/30 — Loss: 14.5708 — Train Acc: 74.71%
Epoch 13/30 — Loss: 14.1432 — Train Acc: 72.13%
Epoch 14/30 — Loss: 12.6871 — Train Acc: 77.87%
Epoch 15/30 — Loss: 12.5680 — Train Acc: 77.01%
Epoch 16/30 — Loss: 12.3135 — Train Acc: 79.31%
Epoch 17/30 — Loss: 11.8006 — Train Acc: 82.47%
Epoch 18/30 — Loss: 11.3187 — Train Acc: 81.90%
Epoch 19/30 — Loss: 11.2940 — Train Acc: 80.17%
Epoch 20/30 — Loss: 10.5464 — Train Acc: 84.20%
Epoch 21/30 — Loss: 8.6320 — Train Acc: 87.93%
Ep

In [58]:
# ---- EVALUATE ----
model.eval()
all_preds = []
all_labels = []

with torch.no_grad():
    for X_batch, y_batch, w_batch in test_loader:
        out = model(X_batch)
        all_preds.extend(out.argmax(1).numpy())
        all_labels.extend(y_batch.numpy())

all_preds  = np.array(all_preds)
all_labels = np.array(all_labels)

acc       = accuracy_score(all_labels, all_preds)
precision = precision_score(all_labels, all_preds, average="weighted", zero_division=0)
f1        = f1_score(all_labels, all_preds, average="weighted", zero_division=0)

print(f"\n{'='*35}")
print(f"  Accuracy  : {acc:.2%}")
print(f"  Precision : {precision:.2%}")
print(f"  F1 Score  : {f1:.2%}")
print(f"{'='*35}")
print("\nPer-class breakdown:\n")
print(classification_report(all_labels, all_preds, labels=np.unique(all_labels), target_names=le.classes_[np.unique(all_labels)], zero_division=0))


  Accuracy  : 86.36%
  Precision : 86.63%
  F1 Score  : 85.76%

Per-class breakdown:

              precision    recall  f1-score   support

      FUTURE       0.94      0.70      0.80        23
    HABITUAL       0.93      0.98      0.96        44
       OTHER       0.00      0.00      0.00         2
        PAST       0.71      0.89      0.79        19

    accuracy                           0.86        88
   macro avg       0.65      0.64      0.64        88
weighted avg       0.87      0.86      0.86        88



In [ ]:
# ---- RULE BIAS AT PREDICTION ----
def rule_bias(sentence, boost=10.0):
    bias = np.zeros(len(label_order))
    label_map = {l: i for i, l in enumerate(label_order)}
    rules = [
    (r'(?:сан|сэн|сон|сөн)$',         "PAST"),
    (r'(?:даг|дэг|дог|дөг)$',         "HABITUAL"),
    (r'(?:на|нэ|но|нө)$',             "FUTURE"),
    (r'[аэоөүию]н$',                  "PAST"),
        ]
    for pattern, label in rules:
        if re.search(pattern, sentence.strip(), re.IGNORECASE):
            if label in label_map:
                bias[label_map[label]] += boost
    return bias

# ---- PREDICT ----
def predict(sentences):
    model.eval()
    encoded = torch.tensor([encode(s) for s in sentences], dtype=torch.long)
    with torch.no_grad():
        logits = model(encoded).numpy()

    results = []
    for i, sentence in enumerate(sentences):
        final = logits[i] + rule_bias(sentence)
        pred  = np.argmax(final)
        results.append(le.inverse_transform([pred])[0])
    return results

In [60]:
# ---- TEST ----
sentences = [
    "Сайн ажилаас гардаг",
    "Тэр хүн хэн юм",
    "Хамгийн муу гэж харсан",
    "Англи бичиг сурна"
]

print("\nPredictions:\n")
for s, label in zip(sentences, predict(sentences)):
    print(f"  {s} -> {label}")


Predictions:

  Сайн ажилаас гардаг -> HABITUAL
  Тэр хүн хэн юм -> FUTURE
  Хамгийн муу гэж харсан -> PAST
  Англи бичиг сурна -> FUTURE
